In [ ]:
import temporaldata as td
import experanto as exp
import InterfaceExperantoTemporaldata.Interface as iet

import sys
sys.path.append("./")

import numpy as np
import h5py

# Temporaldata -> Experanto -> Temporaldata

In [ ]:
def td_check_equality(first: [td.Data, td.RegularTimeSeries, td.IrregularTimeSeries, td.Interval], second: [td.Data, td.RegularTimeSeries, td.IrregularTimeSeries, td.Interval]) -> bool:
    for key in first.keys():
        if key in second.keys():
            if isinstance(getattr(first, key), np.ndarray):
                if (getattr(first, key) != getattr(second, key)).all():
                    print(f"Value mismatch for key '{key}': {getattr(first, key)} != {getattr(second, key)}")
                    return False
            elif isinstance(getattr(first, key), (int, float, str, bool, list, dict, np.bool_)):
                if getattr(first, key) != getattr(second, key):
                    print(f"Value mismatch for key '{key}': {getattr(first, key)} != {getattr(second, key)}")
                    return False
            else:
                if not td_check_equality(getattr(first, key), getattr(second, key)):
                    return False
        else:
            print(f"Key '{key}' not found in second object.")
            return False
    return True
            

In [ ]:
with h5py.File("/Users/robingundlach/Desktop/Bachelor-Arbeit/Temporaldata/data/processed/perich_miller_population_2018/c_20131003_center_out_reaching.h5", "r") as f:
    td_dataset = td.Data.from_hdf5(f, lazy=False)  
print("Dataset: ", td_dataset)


In [ ]:
interface_dict = iet.Interface({"example": td_dataset})
interface_example = interface_dict.process_as_experanto("example")
print("interface_example: ", interface_example)


In [ ]:
interface_dict_backward = iet.Interface({"example": interface_example})
interface_example_backward = interface_dict_backward.process_as_temporaldata("example")
print("interface_example_backward: ", interface_example_backward)


In [ ]:
print("Checking equality of original and backward-converted Temporaldata objects...")
if td_check_equality(td_dataset, interface_example_backward):
    print("Success: The original and backward-converted Temporaldata objects are equal.")
else:    print("Failure: The original and backward-converted Temporaldata objects are NOT equal.")

# Experanto -> Temporaldata -> Experanto

In [ ]:
from experanto.interpolators import SequenceInterpolator, ScreenInterpolator, TimeIntervalInterpolator, SpikeInterpolator
def exp_check_equality(first, second):
    for dev in first.devices.keys():
        if dev in second.devices.keys():
            if isinstance(dev, SequenceInterpolator):
                if not (first.devices[dev].sampling_rate == second.devices[dev].sampling_rate and
                        first.devices[dev].start_time == second.devices[dev].start_time and
                        first.devices[dev].end_time == second.devices[dev].end_time and
                        first.devices[dev].valid_interval == second.devices[dev].valid_interval and
                        first.devices[dev].n_signals == second.devices[dev].n_signals):
                    print(f"Metadata mismatch for device '{dev}'")
                    return False
            if isinstance(dev, ScreenInterpolator):
                if not ((first.devices[dev].timestamps == second.devices[dev].timestamps).all() and
                        first.devices[dev].start_time == second.devices[dev].start_time and
                        first.devices[dev].end_time == second.devices[dev].end_time and
                        first.devices[dev].valid_interval == second.devices[dev].valid_interval and
                        first.devices[dev]._num_frames == second.devices[dev]._num_frames):
                    print(f"Metadata mismatch for device '{dev}'")
                    return False
            if isinstance(dev, TimeIntervalInterpolator):
                if not (first.devices[dev].start_time == second.devices[dev].start_time and
                        first.devices[dev].end_time == second.devices[dev].end_time and
                        first.devices[dev].valid_interval == second.devices[dev].valid_interval):
                    print(f"Metadata mismatch for device '{dev}'")
                    return False
                if hasattr(first.devices[dev], "labeled_intervals") and hasattr(second.devices[dev], "labeled_intervals"):
                    if not first.devices[dev].labeled_intervals == second.devices[dev].labeled_intervals:
                        print(f"Metadata mismatch for device '{dev}'")
                        return False
                elif hasattr(first.devices[dev], "labeled_intervals") ^ hasattr(second.devices[dev], "labeled_intervals"):
                    print("No labeled_intervals attribute in one of the TimeIntervalInterpolators")
                    return False
            if isinstance(dev, SpikeInterpolator):
                if not (first.devices[dev].start_time == first.devices[dev].start_time and
                        first.devices[dev].end_time == first.devices[dev].end_time and
                        first.devices[dev].valid_interval == second.devices[dev].valid_interval and
                        first.devices[dev].n_signals == second.devices[dev].n_signals and
                        first.devices[dev].indices == second.devices[dev].indices and
                        (first.devices[dev].spikes == second.devices[dev].spikes).all):
                    print(f"Metadata mismatch for device '{dev}'")
                    return False
        else:
            print(f"Device '{dev}' not found in second object.")
            return False
    return True

### first example Experanto -> Temporaldata -> Experanto

In [ ]:
from experanto.experiment import Experiment
root_folder = './dynamic29228-2-10-Video-full'
e = Experiment(root_folder)

In [ ]:
interface_dict = iet.Interface({"example": e})
interface_example = interface_dict.process_as_temporaldata("example")

In [ ]:
print("interface_example: ", interface_example)

In [ ]:
interface_dict_backward = iet.Interface({"example": interface_example})
interface_example_backward = interface_dict_backward.process_as_experanto("example")
print("interface_example_backward: ", interface_example_backward)

In [ ]:
print("Checking equality of original and backward-converted Experanto objects...")
if exp_check_equality(e, interface_example_backward):
    print("Success: The original and backward-converted Experanto objects are equal.")
else:    print("Failure: The original and backward-converted Experanto objects are NOT equal.")

### second example Experanto -> Temporaldata -> Experanto

In [ ]:
from experanto.configs import DEFAULT_MODALITY_CONFIG

config = dict(DEFAULT_MODALITY_CONFIG)
config["tiers"] = {
    "interpolation": {
        "_target_": "experanto.interpolators.TimeIntervalInterpolator"
    }
}
config["calcium"] = {
    "interpolation": {
        "_target_": "experanto.interpolators.SequenceInterpolator"
    }
}
e = Experiment(root_folder="./experanto/examples/experanto_000039", modality_config=config)


In [ ]:
interface_dict = iet.Interface({"example": e})
interface_example = interface_dict.process_as_temporaldata("example")

In [ ]:
print("interface_example: ", interface_example)

In [ ]:
interface_dict_backward = iet.Interface({"example": interface_example})
interface_example_backward = interface_dict_backward.process_as_experanto("example")
print("interface_example_backward: ", interface_example_backward)

In [ ]:
print("Checking equality of original and backward-converted Experanto objects...")
if exp_check_equality(e, interface_example_backward):
    print("Success: The original and backward-converted Experanto objects are equal.")
else:    print("Failure: The original and backward-converted Experanto objects are NOT equal.")